In [73]:
!wget -N https://raw.githubusercontent.com/datainpoint/classroom-visualization-and-modern-data-science/main/data/taiwan_election_2024.db

--2026-03-31 15:59:06--  https://raw.githubusercontent.com/datainpoint/classroom-visualization-and-modern-data-science/main/data/taiwan_election_2024.db
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 45576192 (43M) [application/octet-stream]
Saving to: ‘taiwan_election_2024.db’

taiwan_election_202 100%[===================>]  43.46M  --.-KB/s    in 0.1s    

Last-modified header missing -- time-stamps turned off.
2026-03-31 15:59:06 (303 MB/s) - ‘taiwan_election_2024.db’ saved [45576192/45576192]



In [74]:
import unittest
import pandas as pd
import sqlite3

In [75]:
answer_01 =\
"""
-- BEGIN SOLUTION
SELECT name
FROM sqlite_master
WHERE type = 'table';
-- END SOLUTION
"""

answer_02 =\
"""
-- BEGIN SOLUTION
SELECT et.election_type AS election_type,
       COUNT(c.id) AS number_of_candidate
FROM candidates c
JOIN election_types et
  ON c.election_type_id = et.id
GROUP BY et.election_type;
-- END SOLUTION
"""

answer_03 =\
"""
-- BEGIN SOLUTION
SELECT county,
       COUNT(*) AS number_of_polling_place
FROM districts
GROUP BY county;
-- END SOLUTION
"""

answer_04 =\
"""
-- BEGIN SOLUTION
SELECT county,
       COUNT(*) AS number_of_polling_place
FROM districts
GROUP BY county
HAVING COUNT(*) > 1000;
-- END SOLUTION
"""

answer_05 =\
"""
-- BEGIN SOLUTION
SELECT p.name,
       COUNT(*) AS count
FROM (
    SELECT DISTINCT candidate_id
    FROM regional_legislators
) rl
JOIN candidates c
  ON rl.candidate_id = c.id
JOIN parties p
  ON c.party_id = p.id
GROUP BY p.name;
-- END SOLUTION
"""

answer_06 =\
"""
-- BEGIN SOLUTION
SELECT p.name,
       COUNT(*) AS count
FROM (
    SELECT DISTINCT candidate_id
    FROM regional_legislators
) rl
JOIN candidates c
  ON rl.candidate_id = c.id
JOIN parties p
  ON c.party_id = p.id
GROUP BY p.name
HAVING COUNT(*) >= 10;
-- END SOLUTION
"""

answer_07 =\
"""
-- BEGIN SOLUTION
SELECT
    d.county,
    rl.legislator_region,
    rl.number,
    p.name AS party,
    c.name,
    SUM(rl.votes) AS sum_votes
FROM regional_legislators rl
JOIN districts d
    ON rl.district_id = d.id
JOIN candidates c
    ON rl.candidate_id = c.id
JOIN parties p
    ON c.party_id = p.id
WHERE d.county = '臺北市'
GROUP BY
    d.county,
    rl.legislator_region,
    rl.number,
    p.name,
    c.name
ORDER BY
    rl.legislator_region,
    rl.number;
-- END SOLUTION
"""

answer_08 =\
"""
-- BEGIN SOLUTION
WITH taipei_result AS (
    SELECT
        d.county,
        rl.legislator_region,
        rl.number,
        p.name AS party,
        c.name,
        SUM(rl.votes) AS sum_votes
    FROM regional_legislators rl
    JOIN districts d
        ON rl.district_id = d.id
    JOIN candidates c
        ON rl.candidate_id = c.id
    JOIN parties p
        ON c.party_id = p.id
    WHERE d.county = '臺北市'
    GROUP BY
        d.county,
        rl.legislator_region,
        rl.number,
        p.name,
        c.name
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY legislator_region
            ORDER BY sum_votes DESC
        ) AS rn
    FROM taipei_result
)
SELECT
    county,
    legislator_region,
    number,
    party,
    name,
    sum_votes
FROM ranked
WHERE rn = 1
ORDER BY legislator_region;
-- END SOLUTION
"""

answer_09 =\
"""
-- BEGIN SOLUTION
WITH party_votes AS (
    SELECT p.id,
           p.name,
           SUM(pl.votes) AS total_votes
    FROM party_legislators pl
    JOIN parties p
      ON pl.party_id = p.id
    GROUP BY p.id, p.name
),
ranked_parties AS (
    SELECT name,
           total_votes,
           ROW_NUMBER() OVER (ORDER BY total_votes DESC) AS rn
    FROM party_votes
),
final_result AS (
    SELECT name, total_votes
    FROM ranked_parties
    WHERE rn <= 3

    UNION ALL

    SELECT '其他',
           SUM(total_votes)
    FROM ranked_parties
    WHERE rn > 3
),
percent_calc AS (
    SELECT name,
           total_votes,
           CAST(total_votes AS REAL) / SUM(total_votes) OVER () AS raw_pct
    FROM final_result
),
rounded AS (
    SELECT name,
           ROUND(raw_pct, 4) AS pct
    FROM percent_calc
),
adjustment AS (
    SELECT (0.9999 - SUM(pct)) AS diff
    FROM rounded
)
SELECT r.name,
       CASE
           WHEN r.name = (
               SELECT name FROM rounded ORDER BY pct DESC LIMIT 1
           )
           THEN ROUND(r.pct + a.diff, 4)
           ELSE r.pct
       END AS votes_percentage
FROM rounded r
CROSS JOIN adjustment a
ORDER BY votes_percentage DESC;
-- END SOLUTION
"""

answer_10 =\
"""
-- BEGIN SOLUTION
WITH presidential AS (
    SELECT
        et.election_type AS election_type,
        p.name AS party,
        ROUND(SUM(pr.votes) * 1.0 / (SELECT SUM(votes) FROM presidents), 4) AS votes_percentage
    FROM presidents pr
    JOIN candidates c
        ON pr.candidate_id = c.id
    JOIN parties p
        ON c.party_id = p.id
    JOIN election_types et
        ON pr.election_type_id = et.id
    WHERE p.name IN ('中國國民黨', '台灣民眾黨', '民主進步黨')
    GROUP BY
        et.election_type,
        p.name
),
regional AS (
    SELECT
        et.election_type AS election_type,
        p.name AS party,
        ROUND(SUM(rl.votes) * 1.0 / (SELECT SUM(votes) FROM regional_legislators), 4) AS votes_percentage
    FROM regional_legislators rl
    JOIN candidates c
        ON rl.candidate_id = c.id
    JOIN parties p
        ON c.party_id = p.id
    JOIN election_types et
        ON rl.election_type_id = et.id
    WHERE p.name IN ('中國國民黨', '台灣民眾黨', '民主進步黨')
    GROUP BY
        et.election_type,
        p.name
),
party_leg AS (
    SELECT
        et.election_type AS election_type,
        p.name AS party,
        ROUND(SUM(pl.votes) * 1.0 / (SELECT SUM(votes) FROM party_legislators), 4) AS votes_percentage
    FROM party_legislators pl
    JOIN parties p
        ON pl.party_id = p.id
    JOIN election_types et
        ON pl.election_type_id = et.id
    WHERE p.name IN ('中國國民黨', '台灣民眾黨', '民主進步黨')
    GROUP BY
        et.election_type,
        p.name
)
SELECT
    election_type,
    party,
    votes_percentage
FROM (
    SELECT election_type, party, votes_percentage FROM party_leg
    UNION ALL
    SELECT election_type, party, votes_percentage FROM regional
    UNION ALL
    SELECT election_type, party, votes_percentage FROM presidential
)
ORDER BY
    CASE election_type
        WHEN '不分區立委' THEN 1
        WHEN '區域立委' THEN 2
        WHEN '總統/副總統' THEN 3
    END,
    CASE party
        WHEN '中國國民黨' THEN 1
        WHEN '台灣民眾黨' THEN 2
        WHEN '民主進步黨' THEN 3
    END;
-- END SOLUTION
"""

In [76]:
class TestAssignmentThree(unittest.TestCase):
    def test_01(self):
        result_01 = pd.read_sql(answer_01, connection)
        self.assertEqual(result_01.shape, (10, 1))
        tables = result_01.iloc[:, 0].values.tolist()
        self.assertIn("aboriginal_legislators", tables)
        self.assertIn("districts", tables)
        self.assertIn("parties", tables)
        self.assertIn("polling_places", tables)
        self.assertIn("regional_legislators", tables)
    def test_02(self):
        result_02 = pd.read_sql(answer_02, connection)
        self.assertEqual(result_02.shape, (4, 2))
        number_of_candidates = result_02.iloc[:, 1].values.tolist()
        self.assertIn(3, number_of_candidates)
    def test_03(self):
        result_03 = pd.read_sql(answer_03, connection)
        self.assertEqual(result_03.shape, (22, 2))
        counties = result_03.iloc[:, 0].values.tolist()
        self.assertIn("新北市", counties)
        self.assertIn("臺中市", counties)
        self.assertIn("臺南市", counties)
        self.assertIn("屏東縣", counties)
        self.assertIn("嘉義縣", counties)
        self.assertIn("嘉義市", counties)
    def test_04(self):
        result_04 = pd.read_sql(answer_04, connection)
        self.assertEqual(result_04.shape, (7, 2))
        counties = result_04.iloc[:, 0].values.tolist()
        self.assertIn("臺北市", counties)
        self.assertIn("臺南市", counties)
        self.assertIn("新北市", counties)
        self.assertIn("桃園市", counties)
        self.assertIn("臺中市", counties)
        self.assertIn("高雄市", counties)
        self.assertIn("彰化縣", counties)
    def test_05(self):
        result_05 = pd.read_sql(answer_05, connection)
        self.assertEqual(result_05.shape, (33, 2))
        parties = result_05.iloc[:, 0].values.tolist()
        self.assertIn("復康聯盟黨", parties)
        self.assertIn("台灣麻將最大黨", parties)
        self.assertIn("社會民主黨", parties)
        self.assertIn("小民參政歐巴桑聯盟", parties)
        self.assertIn("台灣綠黨", parties)
        self.assertIn("台灣基進", parties)
    def test_06(self):
        result_06 = pd.read_sql(answer_06, connection)
        parties = result_06.iloc[:, 0].values.tolist()
        self.assertEqual(result_06.shape, (11, 2))
        self.assertIn("制度救世島", parties)
        self.assertIn("臺灣雙語無法黨", parties)
        self.assertIn("司法改革黨", parties)
        self.assertIn("小民參政歐巴桑聯盟", parties)
        self.assertIn("台灣維新", parties)
        self.assertIn("人民最大黨", parties)
    def test_07(self):
        result_07 = pd.read_sql(answer_07, connection)
        self.assertEqual(result_07.shape, (53, 6))
    def test_08(self):
        result_08 = pd.read_sql(answer_08, connection)
        self.assertEqual(result_08.shape, (8, 6))
        parties = result_08.iloc[:, 3].values.tolist()
        self.assertIn("民主進步黨", parties)
        self.assertIn("中國國民黨", parties)
        elected = result_08.iloc[:, 4].values.tolist()
        self.assertIn("吳思瑤", elected)
        self.assertIn("王世堅", elected)
        self.assertIn("吳沛憶", elected)
        self.assertIn("王鴻薇", elected)
        self.assertIn("李彥秀", elected)
        self.assertIn("賴士葆", elected)
    def test_09(self):
        result_09 = pd.read_sql(answer_09, connection)
        self.assertEqual(result_09.shape, (4, 2))
        parties = result_09.iloc[:, 0].values.tolist()
        self.assertIn("民主進步黨", parties)
        self.assertIn("中國國民黨", parties)
        self.assertIn("台灣民眾黨", parties)
        self.assertIn("其他", parties)
        percentages = result_09.iloc[:, 1].values.tolist()
        self.assertLessEqual(sum(percentages), 1.0)
    def test_10(self):
        result_10 = pd.read_sql(answer_10, connection)
        self.assertEqual(result_10.shape, (9, 3))
        parties = result_10.iloc[:, 1].values.tolist()
        self.assertIn("民主進步黨", parties)
        self.assertIn("中國國民黨", parties)
        self.assertIn("台灣民眾黨", parties)

connection = sqlite3.connect("taiwan_election_2024.db")
suite = unittest.TestLoader().loadTestsFromTestCase(TestAssignmentThree)
runner = unittest.TextTestRunner(verbosity=2)
test_results = runner.run(suite)
number_of_failures = len(test_results.failures)
number_of_errors = len(test_results.errors)
number_of_test_runs = test_results.testsRun
number_of_successes = number_of_test_runs - (number_of_failures + number_of_errors)
print(f"You've got {number_of_successes} successes among {number_of_test_runs} questions.")

test_01 (__main__.TestAssignmentThree.test_01) ... ok
test_02 (__main__.TestAssignmentThree.test_02) ... ok
test_03 (__main__.TestAssignmentThree.test_03) ... ok
test_04 (__main__.TestAssignmentThree.test_04) ... ok
test_05 (__main__.TestAssignmentThree.test_05) ... ok
test_06 (__main__.TestAssignmentThree.test_06) ... ok
test_07 (__main__.TestAssignmentThree.test_07) ... ok
test_08 (__main__.TestAssignmentThree.test_08) ... ok
test_09 (__main__.TestAssignmentThree.test_09) ... ok
test_10 (__main__.TestAssignmentThree.test_10) ... ok

----------------------------------------------------------------------
Ran 10 tests in 0.993s

OK


You've got 10 successes among 10 questions.
